# [Day 06] PyTorch BERT 파인튜닝 & TorchServe 실시간 모델 서빙

> **수업 범위**: 「모두를 위한 MLOps」 17강(모델 학습 및 서빙 도입) ~ 18강(Vertex AI 소개)  
> **핵심 목표**: 자연어 처리(NLP) 사전학습 모델인 BERT를 4가지 카테고리 뉴스 기사 데이터셋(AG News)으로 파인튜닝하고, 가중치와 핸들러를 .mar 파일로 아카이빙하여 TorchServe 기반 고성능 실시간 추론 API 서버를 구축합니다.

---

### [실습 파이프라인 구조]
1. [환경 준비] T4 GPU 런타임 점검 및 필수 패키지 설치
2. [데이터 준비] AG News (4분류) 데이터셋 로드 & BERT Tokenizer 전처리
3. [모델 학습] BertForSequenceClassification 3 Epochs 파인튜닝
4. [모델 평가] 손실 곡선(Loss), 정확도(Accuracy 85%+), 단건 추론, 혼동 행렬(Confusion Matrix)
5. [서빙 패키징] 가중치(.pth) 저장 + 핸들러(model_handler.py) 작성 + .mar 아카이빙
6. [서버 배포] TorchServe 추론 서버 기동(5000번 포트) + /ping 헬스체크 + 실시간 예측 API 호출
7. [엔터프라이즈 MLOps] Google Cloud Vertex AI 플랫폼 아키텍처 비교 정리 (18강)

## 1. 패키지 설치 및 GPU 런타임 점검 (STEP 1)

### [설명]
* **torchserve & torch-model-archiver**: PyTorch 재단에서 공식 개발한 고성능 모델 서빙 프레임워크 및 배포용 .mar 아카이브 생성 도구입니다.
* **transformers & datasets**: HuggingFace의 사전학습 모델 및 표준 벤치마크 데이터셋 라이브러리입니다.
* **런타임 점검**: 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 T4 GPU가 켜져 있는지 확인합니다.

In [ ]:
# 1. 필수 라이브러리 설치 (datasets 3.1.0 고정, 최신 torchserve 패키지 설치)
!pip install -q datasets==3.1.0 torchserve torch-model-archiver nvgpu

import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA GPU 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 디바이스 이름: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리(VRAM): {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 데이터셋 로드 & BERT 모델 및 데이터로더 준비 (STEP 2 & 3)

### [설명]
* **AG News 4분류 데이터셋**: 뉴스 기사를 4가지 주제로 분류하는 벤치마크 데이터셋입니다.
  * 0: World (시사/세계)
  * 1: Sports (스포츠)
  * 2: Business (경제/비즈니스)
  * 3: Sci/Tech (기술/과학)
* **BertTokenizer (bert-base-uncased)**: 영문 텍스트를 서브워드(Subword) 단위 토큰으로 쪼개고, 모델이 이해할 수 있는 정수 ID(input_ids)와 실제 토큰 위치를 나타내는 attention_mask를 생성합니다.
* **BertForSequenceClassification**: 사전학습된 12계층 BERT Transformer 인코더 위에 4개 클래스를 분류하는 선형 레이어(Classification Head)가 얹어진 모델입니다.
* **AdamW**: 가중치 감쇠(Weight Decay)를 올바르게 적용한 트랜스포머 전용 옵티마이저입니다 (학습률 lr=5e-5).
* **NewsDataset & collate_fn**: datasets 라이브러리와 최신 torchvision 간의 버전 충돌을 방지하기 위해 표준 PyTorch Dataset 형태로 안전하게 래핑합니다.

In [ ]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import confusion_matrix

# 1. HuggingFace 허브에서 AG News 데이터셋 다운로드
print("[데이터 로드] AG News 데이터셋 다운로드 중...")
raw_dataset = load_dataset("fancyzhx/ag_news")

# 2. 사전학습된 BERT 토크나이저 및 4분류 모델 초기화
print("[모델 초기화] BERT 토크나이저 및 4분류 모델 로드 중...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
optimizer = AdamW(model.parameters(), lr=5e-5)

# 3. PyTorch Dataset 클래스 정의 (데이터 접근 표준화)
class NewsDataset(Dataset):
    def __init__(self, hf_data):
        self.texts = list(hf_data["text"])
        self.labels = list(hf_data["label"])

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

# 4. 배치 단위 토크나이징 Collate 함수 정의 (동적 패딩 & 텐서 변환)
def collate_fn(batch):
    texts = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    # max_length 512 기준 패딩 및 텐서 반환
    inputs = tokenizer(texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
    inputs["label"] = labels
    return inputs

# 5. 빠른 실습을 위해 학습용 1,200건, 테스트용 800건 추출
train_sub = raw_dataset["train"].select(range(1200))
test_sub = raw_dataset["test"].select(range(800))

# 6. 배치 크기 8 단위 데이터로더 생성 (에폭당 150 스텝)
train_loader = DataLoader(NewsDataset(train_sub), batch_size=8, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(NewsDataset(test_sub), batch_size=8, shuffle=False, collate_fn=collate_fn)

print("[완료] 데이터로더 및 BERT 모델 준비 완료!")

## 3. GPU 파인튜닝 학습 루프 실행 (STEP 4)

### [설명]
* **학습 메커니즘**:
  1. **순전파(Forward)**: 입력 텍스트 토큰을 BERT 모델에 넣어 예측 Logits와 CrossEntropy Loss를 계산합니다.
  2. **역전파(Backward)**: loss.backward()로 각 가중치에 대한 기울기(Gradient)를 역전파합니다.
  3. **가중치 갱신(Optimizer Step)**: optimizer.step()으로 가중치를 최적화하고 optimizer.zero_grad()로 이전 기울기를 초기화합니다.
* **에폭 수**: 3 에폭 (150 스텝 * 3 = 총 450 스텝, T4 GPU 기준 약 1~2분 소요)

In [ ]:
# 1. 모델을 GPU 메모리로 전송
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"[학습 디바이스] {device}")

# 2. 학습 파라미터 및 손실 기록용 리스트 초기화
num_epochs = 3
losses = []

# 3. 3 에폭 학습 루프 실행
for epoch in range(num_epochs):
    model.train()  # 모델을 학습 모드로 설정 (Dropout 활성화)
    total_loss = 0
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        # 배치 텐서를 GPU로 이동
        inputs = {key: batch[key].to(device) for key in batch}
        labels = inputs.pop("label")
        
        # 기울기 초기화
        optimizer.zero_grad()
        
        # 순전파 및 손실 계산
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        
        # 역전파 및 가중치 업데이트
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        losses.append(loss.item())
        
    # 에폭별 평균 손실 출력 (점진적으로 0.6 -> 0.3 -> 0.15 로 하락)
    average_loss = total_loss / len(train_loader)
    print(f"-> Epoch {epoch + 1} 완료 | 평균 손실(Average Loss): {average_loss:.4f}")

print("\n[학습 완료] BERT 모델 파인튜닝 학습이 성공적으로 끝났습니다.")

## 4. 다각도 모델 평가 & 시각화 (STEP 5 ~ 8)

### [설명]
학습이 끝난 모델이 실무에서 쓸 만한지 검증하기 위해 4가지 관점에서 평가합니다:
1. **학습 손실 곡선 (Loss Curve)**: 스텝이 진행됨에 따라 오차가 우하향하며 수렴했는지 확인합니다.
2. **테스트셋 정확도 (Accuracy)**: 학습에 쓰이지 않은 800건의 새 뉴스에 대해 85% 안팎의 높은 분류 성능을 내는지 평가합니다.
3. **단건 기사 실시간 예측 (Inference)**: 모델이 뱉는 Logits -> Softmax(확률 변환) -> Argmax(최대 확률 클래스 인덱스 선택) 과정을 통해 축구 감독 사임 기사를 Sports(1)로 맞추는지 확인합니다.
4. **혼동 행렬 (Confusion Matrix)**: 대각선(정답) 외에 어떤 카테고리가 어디로 오분류되는지 히트맵으로 시각화 분석합니다.

In [ ]:
# 1. [STEP 5] 학습 손실 곡선(Loss Curve) 시각화
plt.figure(figsize=(10, 4))
plt.plot(losses, color="#4285f4", linewidth=2)
plt.xlabel("Step (총 450 스텝)")
plt.ylabel("CrossEntropy Loss")
plt.title("Training Loss per Step Across Epochs")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

# 2. [STEP 6] 테스트셋 800건에 대한 정확도(Accuracy) 평가
model.eval()  # 평가 모드로 전환 (Dropout 비활성화)
all_predictions = []
all_labels = []

with torch.no_grad():  # 평가 시 기울기 계산 비활성화 (메모리 절약 & 속도 향상)
    for batch in tqdm(test_loader, desc="Evaluating"):
        inputs = {key: batch[key].to(device) for key in batch}
        labels = inputs.pop("label")
        
        outputs = model(**inputs)
        predicted_labels = torch.argmax(outputs.logits, dim=1)
        
        all_predictions.extend(predicted_labels.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = (np.array(all_predictions) == np.array(all_labels)).mean()
print(f"\n[평가 결과] 테스트셋 정확도: {accuracy * 100:.2f}%")

# 3. [STEP 7] 실제 임의의 뉴스 기사 단건 예측 테스트
labeling_mapper = ["0: World (시사)", "1: Sports (스포츠)", "2: Business (경제)", "3: Sci/Tech (기술·과학)"]
sample_news = "[Official] 'Legendary Coach Resigns -> Appoints New Commander' Suwon Completes Coaching Staff... Scout Bae Ki-jong Joins"
sample_inputs = tokenizer(sample_news, truncation=True, padding="max_length", return_tensors="pt").to(device)

with torch.no_grad():
    sample_logits = model(**sample_inputs).logits
    sample_prob = F.softmax(sample_logits, dim=1)
    pred_idx = torch.argmax(sample_logits, dim=1).item()

print(f"\n[입력 뉴스 본문] {sample_news}")
print(f"[예측 카테고리] {labeling_mapper[pred_idx]} (확신도: {sample_prob[0][pred_idx].item()*100:.2f}%)")

# 4. [STEP 8] 혼동 행렬(Confusion Matrix) 히트맵 시각화
conf_matrix = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap=sns.light_palette("#4285f4", as_cmap=True),
            xticklabels=["World", "Sports", "Business", "Sci/Tech"],
            yticklabels=["World", "Sports", "Business", "Sci/Tech"])
plt.xlabel("Predicted Label (예측값)")
plt.ylabel("True Label (실제 정답)")
plt.title("Confusion Matrix Heatmap")
plt.show()

## 5. 서빙을 위한 모델 패키징 & `.mar` 아카이빙 (STEP 9 ~ 12)

### [왜 .pth 파일만으로는 서빙할 수 없는가?]
* .pth 파일은 모델의 순수 가중치(수치 파라미터)만 가지고 있습니다.
* 프로덕션 API 서버로 서빙하려면 **[가중치(.pth) + 모델 아키텍처 + 전후처리 파이썬 핸들러(model_handler.py) + 설정 파일(config.properties) + 부속 토큰 사전(vocab.txt)]**을 하나로 압축 묶음한 **.mar (Model Archive)** 파일이 반드시 필요합니다.

### [TorchServe 핸들러 4단계 생명주기 (model_handler.py)]
1. **initialize()**: 서버 시작 시 1회 실행되어 모델 아키텍처를 만들고, .pth 가중치를 주입하고, 추론 전용 eval() 모드로 전환합니다.
2. **preprocess()**: 외부 사용자가 HTTP POST로 보낸 JSON 요청에서 텍스트를 추출하고, 토크나이저로 텐서 변환하여 GPU로 보냅니다.
3. **inference()**: with torch.no_grad() 환경에서 모델 순전파를 실행하여 Logits를 계산합니다.
4. **postprocess()**: Logits를 Softmax 확률로 변환하고 가장 가능성이 높은 클래스 라벨과 확률값을 JSON 응답으로 가공합니다.

In [ ]:
# 1. [STEP 9] 학습된 모델의 가중치 딕셔너리(.pth) 파일 저장
model_save_path = "bert_news_classification_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"[저장 완료] 가중치 파일(.pth): {model_save_path}")

In [ ]:
# 2. [STEP 10] TorchServe 커스텀 전후처리 핸들러 파일 생성 (model_handler.py)
%%writefile model_handler.py
import json
import torch
from ts.context import Context
from ts.torch_handler.base_handler import BaseHandler
from transformers import BertTokenizer, BertForSequenceClassification

class ModelHandler(BaseHandler):
    def __init__(self):
        self.initialized = False
        self.tokenizer = None
        self.model = None

    def initialize(self, context: Context):
        """[1단계] 서버 기동 시 1회 호출: 모델 아키텍처 생성, 가중치 로드, GPU 할당, eval() 전환"""
        self.initialized = True
        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
        self.model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
        self.model.load_state_dict(torch.load("bert_news_classification_model.pth", map_location="cpu"))
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def preprocess(self, data: list[dict]):
        """[2단계] 요청 수신마다 호출: JSON 파싱, 기사 텍스트 추출, BERT 토큰화"""
        model_input_texts = []
        for item in data:
            body = item.get("data") or item.get("body")
            if isinstance(body, (bytes, bytearray)):
                body = body.decode("utf-8")
            if isinstance(body, str):
                try:
                    body = json.loads(body)
                except:
                    pass
            if isinstance(body, dict) and "data" in body:
                model_input_texts.extend(body["data"])
            elif isinstance(body, list):
                model_input_texts.extend(body)
            elif isinstance(body, str):
                model_input_texts.append(body)

        inputs = self.tokenizer(model_input_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
        return inputs.to(self.device)

    def inference(self, input_batch):
        """[3단계] 추론 실행: no_grad 상태에서 Logits 출력"""
        with torch.no_grad():
            outputs = self.model(**input_batch)
        return outputs.logits

    def postprocess(self, inference_output):
        """[4단계] 후처리: Softmax 확률 변환 후 최고 확률 레이블 번호와 확률값 JSON 반환"""
        probabilities = torch.nn.functional.softmax(inference_output, dim=1)
        results = []
        for prob in probabilities:
            label_idx = int(torch.argmax(prob))
            results.append({
                "label": label_idx,
                "probability": float(prob.max().item())
            })
        return results


In [ ]:
# 3. [STEP 11] TorchServe 포트 설정 파일 생성 (config.properties)
# Colab의 기존 8000번대 포트 충돌을 피하기 위해 5000/5001/5002로 재지정
%%writefile config.properties
inference_address=http://0.0.0.0:5000
management_address=http://0.0.0.0:5001
metrics_address=http://0.0.0.0:5002


In [ ]:
# 4. [STEP 12] vocab 부속 파일 다운로드 & torch-model-archiver 로 .mar 아카이빙 패키징
!wget -q https://raw.githubusercontent.com/microsoft/SDNet/master/bert_vocab_files/bert-base-uncased-vocab.txt -O bert-base-uncased-vocab.txt
!mkdir -p model-store

# .mar 파일 생성 CLI 실행
!torch-model-archiver \
  --model-name bert_news_classification \
  --version 1.0 \
  --serialized-file bert_news_classification_model.pth \
  --handler ./model_handler.py \
  --extra-files "bert-base-uncased-vocab.txt" \
  --export-path model-store \
  -f

!ls -lh model-store/bert_news_classification.mar
print("[패키징 완료] model-store/bert_news_classification.mar 배포 패키지 생성 완료!")

## 6. TorchServe 추론 서버 기동 & 실시간 예측 API 호출 (STEP 13 ~ 15)

### [설명]
* **torchserve --start**: 백그라운드 프로세스로 서빙 엔진을 가동하고 .mar 아카이브를 로드합니다.
* **/ping**: 서버와 모델이 정상 로드되어 요청을 받을 준비가 되었는지 확인하는 헬스체크 엔드포인트입니다.
* **/predictions/{model_name}**: 클라이언트가 텍스트 기사 JSON을 전송하면 실시간으로 카테고리를 추론하여 응답하는 REST API 엔드포인트입니다.

In [ ]:
# 1. [STEP 13] TorchServe 서버 백그라운드 기동
%%script bash --bg
torchserve --start --ncs \
  --ts-config config.properties \
  --model-store model-store \
  --models bert_news_classification=bert_news_classification.mar \
  --disable-token-auth


In [ ]:
# 2. [STEP 13 헬스체크] 모델 로드 대기 (15초) 후 서버 상태 확인
import time
print("[기동 대기] TorchServe 모델 메모리 로딩 대기 중 (15초)...")
time.sleep(15)

# /ping 엔드포인트 호출 (기대 응답: {"status": "Healthy"})
!curl -s -X GET http://localhost:5000/ping

In [ ]:
# 3. [STEP 14] 테스트용 3가지 주제 뉴스 기사 JSON 페이로드 생성
# 최상위 키는 핸들러가 읽는 'data' 이며 값은 문자열 배열입니다.
sports_json = {"data": ["Bleary-eyed from 16 hours on a Greyhound bus, he strolled into the baseball stadium ready for the championship game."]}
business_json = {"data": ["DETROIT — Automotive stocks surged as Wall Street reported record quarterly profits today."]}
sci_tech_json = {"data": ["OpenVoice comprises two AI models working together for text-to-speech conversion and neural network voice synthesis."]}

with open("request_sports.json", "w") as f:
    json.dump(sports_json, f)
with open("request_business.json", "w") as f:
    json.dump(business_json, f)
with open("request_sci_tech.json", "w") as f:
    json.dump(sci_tech_json, f)

print("[생성 완료] 스포츠, 비즈니스, 기술 기사 JSON 파일 3종 생성 완료!")

In [ ]:
# 4. [STEP 15] 각각의 뉴스 기사를 TorchServe API로 전송하여 실시간 분류 예측 결과 확인!
print("="*60)
print("[1. 스포츠 기사 예측 요청] (/predictions/bert_news_classification):")
!curl -s -X POST -H "Content-Type: application/json" -T "request_sports.json" http://localhost:5000/predictions/bert_news_classification

print("\n\n[2. 경제 기사 예측 요청]:")
!curl -s -X POST -H "Content-Type: application/json" -T "request_business.json" http://localhost:5000/predictions/bert_news_classification

print("\n\n[3. 기술·과학 기사 예측 요청]:")
!curl -s -X POST -H "Content-Type: application/json" -T "request_sci_tech.json" http://localhost:5000/predictions/bert_news_classification
print("\n" + "="*60)
print("[검증 완료] 0: World, 1: Sports, 2: Business, 3: Sci/Tech 각각 정확하게 판별되었습니다.")

## 7. [18강 핵심 요약] Google Cloud Vertex AI 엔터프라이즈 MLOps 플랫폼

### [노트북 한 장으로 끝낸 수동 파이프라인의 한계점]
방금 실습은 연구자의 노트북 안에서 학습부터 서빙까지 수동으로 진행했습니다. 이를 실제 회사 프로덕션 서비스에 적용하려면 다음과 같은 문제들을 해결해야 합니다:
1. **학습이 개인 장비에 묶임**: 매번 개인 컴퓨터 GPU를 점유해야 함
2. **모델 파일이 로컬에 산재**: 버전(v1, v2)과 성능 메트릭이 체계적으로 관리되지 않아 롤백이 어려움
3. **서빙 서버를 수동으로 띄움**: 서버 다운 시 복구 및 트래픽 폭증 시 오토스케일링 불가
4. **피처 데이터 분산**: 학습에 쓰인 데이터와 서빙 로그가 분리되어 모델 드리프트(Drift) 감지 불가

---

### [Vertex AI 대응 컴포넌트 총괄표]

| 수동 파이프라인의 한계 | Vertex AI 대응 컴포넌트 | 엔터프라이즈 MLOps 효과 |
| :--- | :--- | :--- |
| **학습이 개인 장비에 묶임** | **Vertex AI Training** | 필요할 때만 클라우드 고성능 GPU를 할당받아 학습하고 끝나면 즉시 반환 (비용 최적화) |
| **가중치 파일의 버전 미관리** | **Model Registry** | 모델의 버전별 아티팩트와 평가 성능 지표를 중앙 저장소에서 체계적 추적 및 롤백 지원 |
| **서빙 서버를 수동 관리** | **Online Prediction** | 모델 등록 시 엔드포인트 URL이 자동 부여되며, 트래픽에 따른 자동 확장(Auto-scaling) 지원 |
| **피처 데이터 분산 및 드리프트** | **Feature Store / Monitoring** | 실시간 피처를 중앙에서 일관 관리하고, 배포 후 데이터 분포 변화(Drift)를 조기에 감지 |
| **학습~배포 단계의 수동 실행** | **Vertex AI Pipelines** | 데이터 수집 -> 가공 -> 학습 -> 평가 -> 배포 전 과정을 하나의 자동화 파이프라인으로 오케스트레이션 |